# Experimento: proteção dos corredores de migração

**Hipótese.** Se um enlace do corredor de migração de um cooperante é usado *só* por membros de $K$, restringir o tráfego de fundo nele durante a janela de alerta libera espectro para a migração, sem entregar esse espectro a quem não coopera.

**Modelo de conhecimento (validação).** Um mediador divulga $\sigma(e)$, quantos ISPs usam cada enlace. Os cooperantes compartilham seus overlays, então contam quantos deles usam $e$; se a contagem é igual a $\sigma(e)$, o enlace é exclusivo de $K$. Se os resultados forem positivos, o próximo passo é trocar $\sigma(e)$ por inferência a partir da ocupação do espectro.

**Onde e quando.** Só em enlaces exclusivos que estão nos 3 primeiros caminhos candidatos de migração de algum cooperante (recalculados a cada entrada em $K$), e só entre a reação e $t_d$. A migração nunca é restringida.

| Variante | O que faz |
|---|---|
| `pesos` | referência: só os pesos α/β/γ (resultado v2 já rodado) |
| `reserva` | o tráfego de fundo não pode usar os 25% de slots do topo nos enlaces protegidos |
| `adaptativo` | o tráfego de fundo é mantido fora do enlace com probabilidade $p = \min(0{,}9,\ \text{bloqueio recente da migração dos donos})$; pode desviar por outro caminho |
| `*_isolado` | controle: cada ISP que reagiu protege os enlaces que só ele usa ($\sigma(e)=1$), cooperando ou não |

A diferença entre `reserva` e `reserva_isolado` (mesmo $K$) é o que a cooperação acrescenta; a diferença entre `reserva_isolado` e `pesos` é o que o mediador sozinho acrescenta.

Todas as variantes usam modulação por distância e o roteador v2.

In [ ]:
import json
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from experiment_worker import run_cooperation_experiment
from run_cooperation_experiment import PROTECOES, consolidar, montar_tarefas, pasta_saida

%matplotlib inline

## Parâmetros

In [ ]:
BASE_INDICES = [0]
VARIANTES = ["reserva", "adaptativo", "reserva_isolado", "adaptativo_isolado"]
MAX_WORKERS = min(os.cpu_count() or 4, 8)
REFERENCIA = pasta_saida(None)            # 'pesos': output/cooperation_experiment_v2

tarefas = []
for nome in VARIANTES:
    pasta = pasta_saida(None, nome)
    pasta.mkdir(parents=True, exist_ok=True)
    for t in montar_tarefas(BASE_INDICES, pasta, first_fit=False):
        t.update(output_dir=str(pasta), protecao=PROTECOES[nome], variante_protecao=nome)
        tarefas.append(t)
for idx, t in enumerate(tarefas):
    t.update(run_index=idx, total_runs=len(tarefas))
print(f"{len(tarefas)} rodadas pendentes | variantes {VARIANTES} | {MAX_WORKERS} processos")
if not (REFERENCIA / "resumo_por_rodada.csv").exists():
    print(f"⚠️ Referência {REFERENCIA} não encontrada: rode Experiment_cooperation_subsets.ipynb com MODULACAO_FIXA = None")

## Rodar

Cada rodada grava um `metrics_*.json`; se interromper, rode de novo e as prontas são puladas.

In [ ]:
inicio = datetime.now()
falhas = []
if tarefas:
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(run_cooperation_experiment, t): t for t in tarefas}
        for n, future in enumerate(as_completed(futures), start=1):
            r = future.result()
            nome = futures[future]["variante_protecao"]
            estado = "✓" if r["success"] else "✗"
            print(f"[{n}/{len(tarefas)}] {estado} {nome} base={r['base_index']} {r['K']}  ({datetime.now() - inicio})")
            if not r["success"]:
                falhas.append(r)
print(f"Duração: {datetime.now() - inicio} | falhas: {len(falhas)}")
for nome in VARIANTES:
    consolidar(pasta_saida(None, nome))

## Carregar

In [ ]:
def carregar(pasta, nome):
    r = pd.read_csv(pasta / "resumo_por_rodada.csv")
    p = pd.read_csv(pasta / "resumo_por_isp.csv")
    r = r[(r["variante"] == "cooperacao") & r["base_index"].isin(BASE_INDICES)].assign(cenario=nome)
    p = p[(p["variante"] == "cooperacao") & p["base_index"].isin(BASE_INDICES)].assign(cenario=nome)
    return r, p

partes = [carregar(REFERENCIA, "pesos")] + [carregar(pasta_saida(None, n), n) for n in VARIANTES]
resumo = pd.concat([r for r, _ in partes], ignore_index=True)
por_isp = pd.concat([p for _, p in partes], ignore_index=True)
ORDEM = ["pesos"] + VARIANTES
I = por_isp["isp_id"].nunique()
resumo.groupby("cenario").size().reindex(ORDEM)

## Visão geral

Média sobre todos os conjuntos $K$, e os extremos $K=\emptyset$ e $K=\mathcal{I}$.

In [ ]:
cols = ["eta_todos", "B_m", "B_n", "B_n_janela"]
geral = resumo.groupby("cenario")[cols].mean().reindex(ORDEM)
extremos = (
    resumo[resumo["k"].isin([0, I])]
    .pivot_table(index="cenario", columns="k", values="eta_todos")
    .reindex(ORDEM)
    .rename(columns={0: "eta K=∅", I: "eta K=I"})
)
diag = resumo.groupby("cenario")[["prot_max_links", "prot_normais_restringidas"]].mean().reindex(ORDEM)
pd.concat([geral, extremos, diag], axis=1).round(4)

## Efeito por conjunto $K$, pareado com a referência

Para cada $K$: variante − `pesos`. Positivo em `d_eta_todos` = mais migração no total.

In [ ]:
ref = resumo[resumo["cenario"] == "pesos"].set_index(["base_index", "K"])
linhas = []
for nome in VARIANTES:
    v = resumo[resumo["cenario"] == nome].set_index(["base_index", "K"])
    d = (v[["eta_todos", "eta_K", "eta_isolados", "B_n", "B_m", "B_n_K"]]
         - ref[["eta_todos", "eta_K", "eta_isolados", "B_n", "B_m", "B_n_K"]])
    d = d.add_prefix("d_").assign(cenario=nome, k=v["k"])
    linhas.append(d.reset_index())
delta = pd.concat(linhas, ignore_index=True)
delta.groupby(["cenario", "k"])[["d_eta_todos", "d_eta_K", "d_eta_isolados", "d_B_n_K", "d_B_m"]].mean().round(4)

In [ ]:
resumo_delta = delta.groupby("cenario").agg(
    ganho_medio=("d_eta_todos", "mean"),
    melhor=("d_eta_todos", "max"),
    pior=("d_eta_todos", "min"),
    K_que_melhoram=("d_eta_todos", lambda s: int((s > 1e-9).sum())),
    K_que_pioram=("d_eta_todos", lambda s: int((s < -1e-9).sum())),
).reindex(VARIANTES)
resumo_delta.round(4)

## O que a cooperação acrescenta

Mesmo $K$ e mesmo mecanismo: escopo `cooperacao` − controle `isolado`.

In [ ]:
coop_vs_ctrl = []
for modo in ["reserva", "adaptativo"]:
    a = resumo[resumo["cenario"] == modo].set_index(["base_index", "K"])
    b = resumo[resumo["cenario"] == f"{modo}_isolado"].set_index(["base_index", "K"])
    d = (a[["eta_todos", "eta_K", "B_n_K"]] - b[["eta_todos", "eta_K", "B_n_K"]]).add_prefix("d_")
    coop_vs_ctrl.append(d.assign(modo=modo, k=a["k"]).reset_index())
coop_vs_ctrl = pd.concat(coop_vs_ctrl, ignore_index=True)
coop_vs_ctrl.groupby(["modo", "k"])[["d_eta_todos", "d_eta_K", "d_B_n_K"]].mean().round(4)

In [ ]:
CORES = {"pesos": "#52514e", "reserva": "#2a78d6", "adaptativo": "#eb6834",
         "reserva_isolado": "#1baf7a", "adaptativo_isolado": "#e87ba4"}
TRACO = {"pesos": "-", "reserva": "-", "adaptativo": "-",
         "reserva_isolado": "--", "adaptativo_isolado": "--"}
TEXTO, GRADE = "#52514e", "#e5e4df"

def estilo(ax, titulo, ylabel):
    ax.set_title(titulo, loc="left", fontsize=12, color="#0b0b0b")
    ax.set_xlabel("|K| (provedores cooperando)", color=TEXTO)
    ax.set_ylabel(ylabel, color=TEXTO)
    ax.set_xticks(range(I + 1))
    ax.tick_params(colors=TEXTO)
    ax.grid(axis="y", color=GRADE, linewidth=0.8)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, titulo, ylabel in [
    (axes[0], "eta_todos", "Fração migrada (todos os ISPs)", "η médio por |K|"),
    (axes[1], "B_n", "Bloqueio de fundo", "B_n médio por |K|"),
]:
    for nome in ORDEM:
        m = resumo[resumo["cenario"] == nome].groupby("k")[col].mean()
        ax.plot(m.index, m.values, TRACO[nome], color=CORES[nome], linewidth=2,
                marker="o", markersize=8, label=nome)
    estilo(ax, titulo, ylabel)
axes[0].legend(frameon=False, loc="best")
plt.tight_layout()
plt.savefig(Path("output") / "protecao_por_k.png", dpi=150, bbox_inches="tight")
plt.show()

## Por ISP: dentro vs fora de $K$

Média de $\eta_i$ e $B_{n,i}$ nos conjuntos $0<|K|<I$.

In [ ]:
parcial = por_isp[(por_isp["k"] > 0) & (por_isp["k"] < I)]
tab = parcial.pivot_table(index=["isp_id"], columns=["cenario", "em_K"], values="eta_i", aggfunc="mean")
tab = tab.reindex(columns=ORDEM, level=0)
tab.round(4)

## Leitura

- **Positivo de verdade** exige `d_eta_todos > 0` na maioria dos $K$ *e* `reserva`/`adaptativo` acima do respectivo `_isolado`. Se só o controle melhora, o ganho vem do mediador ($\sigma(e)$), não da cooperação.
- Compare o ganho em migração com o custo em `B_n_K` (tráfego de fundo dos próprios cooperantes).
- `prot_max_links = 0` em algum $K$ significa que não havia enlace exclusivo no corredor: a regra nem chegou a agir.
- Um cenário base só: trate como indicativo.